## error propagation: feature profile correlation (expert vs. cellpose)

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (analysis_input, profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 10)

pass  # was os.chdir; the bootstrap above handles paths


In [ ]:
# Treatment and cell line mapping from TheseWells.txt
treatment_map = {
    # HCT116
    ('PB000139', 'C03'): ('HCT116', 'binimetinib'),
    ('PB000140', 'G05'): ('HCT116', 'binimetinib'),
    ('PB000139', 'D11'): ('HCT116', 'binimetinib'),
    ('PB000139', 'G08'): ('HCT116', 'abemaciclib'),
    ('PB000140', 'G08'): ('HCT116', 'abemaciclib'),
    ('PB000140', 'J04'): ('HCT116', 'abemaciclib'),
    ('PB000139', 'C23'): ('HCT116', 'SN-38'),
    ('PB000140', 'C23'): ('HCT116', 'SN-38'),
    ('PB000139', 'J06'): ('HCT116', 'SN-38'),
    ('PB000137', 'D12'): ('HCT116', 'DMSO'),
    ('PB000137', 'F06'): ('HCT116', 'DMSO'),
    ('PB000140', 'N05'): ('HCT116', 'DMSO'),
    # HT29
    ('PB000139', 'C14'): ('HT29', 'binimetinib'),
    ('PB000140', 'D22'): ('HT29', 'binimetinib'),
    ('PB000140', 'I13'): ('HT29', 'binimetinib'),
    ('PB000139', 'J15'): ('HT29', 'abemaciclib'),
    ('PB000140', 'B14'): ('HT29', 'abemaciclib'),
    ('PB000140', 'G19'): ('HT29', 'abemaciclib'),
    ('PB000139', 'D06'): ('HT29', 'SN-38'),
    ('PB000139', 'C12'): ('HT29', 'SN-38'),
    ('PB000140', 'J17'): ('HT29', 'SN-38'),
    ('PB000140', 'H23'): ('HT29', 'DMSO'),
    ('PB000141', 'G02'): ('HT29', 'DMSO'),
    ('PB000141', 'M23'): ('HT29', 'DMSO'),
}

#### Calculate error propegation 

In [ ]:
from scipy.stats import pearsonr

# ─────────────────────────────────────────────────────────────────────────────
# Step 1 – well filter (reuse treatment_map defined above)
# ─────────────────────────────────────────────────────────────────────────────
well_filter = set(treatment_map.keys())   # {(barcode, well), ...}
print(f"Wells to analyse: {len(well_filter)}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 2 – load expert-annotation CSVs
# Plane is encoded in Metadata_Site (values 1,3,5,7,9,11 = z-planes)
# ─────────────────────────────────────────────────────────────────────────────
# 519 MB of CellProfiler output, held outside the repo.
results_dir = external('expert-annotation') / 'results'
CACHED_CORR = analysis_input('3_SupplFigure2/data/error_propagation_cached.csv')
HAVE_RESULTS = results_dir.is_dir()

expert_dfs = {}
for compartment in ['cells', 'nuclei', 'cytoplasm']:
    df = pd.read_csv(results_dir / f'featICF_{compartment}.csv')

    # Plate column (may be duplicated — take first occurrence)
    plate_cols = [c for c in df.columns if c.startswith('Metadata_Plate')]
    df = df.copy()
    df['_plate'] = df[plate_cols[0]]
    df['_well']  = df['Metadata_Well']
    df['_plane'] = df['Metadata_Site'].astype(int)   # z-plane: 1,3,5,7,9,11

    # Filter to TheseWells
    df['_plate_well'] = list(zip(df['_plate'], df['_well']))
    df = df[df['_plate_well'].isin(well_filter)].reset_index(drop=True)

    expert_dfs[compartment] = df
    print(f"Expert {compartment}: {len(df)} cells | planes: {sorted(df['_plane'].unique())}")

# ─────────────────────────────────────────────────────────────────────────────
# Step 3 – load original (cellpose) parquets — HCT116 + HT29
# Metadata_Site = z-plane, 0-indexed (0–12); expert uses odd planes 1,3,5,7,9,11
# ─────────────────────────────────────────────────────────────────────────────
sc_dir = Path(features("exp1_main", "011225", "SingleCell"))

parts = []
for cell_line_file in ['HCT116.parquet', 'HT29.parquet']:
    df = pd.read_parquet(sc_dir / cell_line_file)
    print(f"Loaded {cell_line_file}: {df.shape}")
    parts.append(df)

df_orig = pd.concat(parts, ignore_index=True)
print(f"Combined: {df_orig.shape}")

barcode_col = next(c for c in df_orig.columns if 'Barcode' in c or 'barcode' in c)
well_col    = 'Metadata_Well'
site_col    = 'Metadata_Site'
print(f"Unique plates: {sorted(df_orig[barcode_col].unique())}")
print(f"Unique planes: {sorted(df_orig[site_col].unique())}")

# Filter to TheseWells
df_orig = df_orig.copy()
df_orig['_plate_well'] = list(zip(df_orig[barcode_col], df_orig[well_col]))
df_orig = df_orig[df_orig['_plate_well'].isin(well_filter)].reset_index(drop=True)
df_orig['_plate'] = df_orig[barcode_col]
df_orig['_well']  = df_orig[well_col]
df_orig['_plane'] = df_orig[site_col].astype(int)
print(f"Original after filtering: {len(df_orig)} cells")

# ─────────────────────────────────────────────────────────────────────────────
# Step 4 – feature alignment helpers
# ─────────────────────────────────────────────────────────────────────────────
_SKIP = ('Metadata_', 'ImageNumber', 'ObjectNumber', 'FileName_',
         'PathName_', '_plate', '_well', '_plane', '_plate_well')

def get_expert_features(df):
    return [c for c in df.columns if not any(c.startswith(s) for s in _SKIP)]

def get_orig_feature_map(compartment):
    """Return {stripped_name: original_col} for columns ending with _{compartment}."""
    suffix = f'_{compartment}'
    return {c[:-len(suffix)]: c
            for c in df_orig.columns
            if c.endswith(suffix) and not any(c.startswith(s) for s in _SKIP)}

# Report overlap
for comp in ['cells', 'nuclei', 'cytoplasm']:
    exp_feats = get_expert_features(expert_dfs[comp])
    orig_map  = get_orig_feature_map(comp)
    common    = [f for f in exp_feats if f in orig_map]
    print(f"{comp}: {len(exp_feats)} expert feats | {len(orig_map)} orig feats | {len(common)} common")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 5 – aggregate & correlate
# Expert planes: 1,3,5,7,9,11 (Metadata_Site); parquet planes: 0–12
# Per-plane groups only match on the shared values 1,3,5,7,9,11
# ─────────────────────────────────────────────────────────────────────────────
def compute_correlations(group_by_plane: bool) -> pd.DataFrame:
    records = []
    grp = ['_plate', '_well', '_plane'] if group_by_plane else ['_plate', '_well']

    for compartment in ['cells', 'nuclei', 'cytoplasm']:
        df_exp    = expert_dfs[compartment]
        orig_map  = get_orig_feature_map(compartment)
        exp_feats = get_expert_features(df_exp)
        common    = [f for f in exp_feats if f in orig_map]
        orig_cols = [orig_map[f] for f in common]

        if not common:
            print(f"  {compartment}: no common features – skipped")
            continue

        med_exp  = df_exp.groupby(grp)[common].median()
        med_orig = df_orig.groupby(grp)[orig_cols].median()
        med_orig.columns = common

        shared_idx = med_exp.index.intersection(med_orig.index)
        print(f"  {compartment}: {len(common)} features | {len(shared_idx)} matching groups")

        for idx in shared_idx:
            v_exp  = med_exp.loc[idx].values.astype(float)
            v_orig = med_orig.loc[idx].values.astype(float)
            valid  = np.isfinite(v_exp) & np.isfinite(v_orig)
            if valid.sum() < 10:
                continue

            r, p = pearsonr(v_exp[valid], v_orig[valid])

            if group_by_plane:
                plate, well, plane = idx
            else:
                plate, well = idx
                plane = None

            cell_line, treatment = treatment_map.get((plate, well), ('Unknown', 'Unknown'))

            records.append({
                'plate':       plate,
                'well':        well,
                'plane':       plane,
                'depth_um':    int(plane) * 5 if plane is not None else None,
                'compartment': compartment,
                'cell_line':   cell_line,
                'treatment':   treatment,
                'pearson_r':   r,
                'p_value':     p,
                'n_features':  int(valid.sum()),
            })

    return pd.DataFrame(records)

print("Per-plane correlations …")
df_corr_plane = compute_correlations(group_by_plane=True)
print(f"  {len(df_corr_plane)} records\n")

print("Per-spheroid (well) correlations …")
df_corr_well = compute_correlations(group_by_plane=False)
print(f"  {len(df_corr_well)} records\n")

print(df_corr_plane.groupby('compartment')['pearson_r'].describe().round(3))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 6 – visualise
# ─────────────────────────────────────────────────────────────────────────────

# ── Per-spheroid boxplot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle('Error propagation: Pearson r (expert vs. cellpose profiles)\nMedian per spheroid (well)', fontsize=13)

for ax, cell_line in zip(axes[:2], ['HCT116', 'HT29']):
    data = df_corr_well[df_corr_well['cell_line'] == cell_line]
    sns.boxplot(data=data, x='treatment', y='pearson_r', hue='compartment',
                ax=ax, palette='Set2', width=0.6, fliersize=0)
    sns.stripplot(data=data, x='treatment', y='pearson_r', hue='compartment',
                  ax=ax, palette='Set2', dodge=True, size=6, alpha=0.8, legend=False)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(cell_line)
    ax.set_xlabel('Treatment')
    ax.set_ylabel('Pearson r' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=25)

sns.boxplot(data=df_corr_well, x='compartment', y='pearson_r', hue='cell_line',
            ax=axes[2], palette='Paired', width=0.5, fliersize=0)
sns.stripplot(data=df_corr_well, x='compartment', y='pearson_r', hue='cell_line',
              ax=axes[2], palette='Paired', dodge=True, size=6, alpha=0.8, legend=False)
axes[2].axhline(0, color='k', lw=0.8, ls='--')
axes[2].set_title('By compartment')
axes[2].set_xlabel('Compartment')
plt.tight_layout()
plt.show()

# ── Per-plane line plot ────────────────────────────────────────────────────
# Expert planes: 1,3,5,7,9,11 → depth 5,15,25,35,45,55 µm
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig2.suptitle('Error propagation by depth\nMedian per plane (Metadata_Site × 5 µm)', fontsize=13)

for ax, compartment in zip(axes2, ['cells', 'nuclei', 'cytoplasm']):
    data = df_corr_plane[df_corr_plane['compartment'] == compartment]
    for cell_line, ls in [('HCT116', '-'), ('HT29', '--')]:
        sub = data[data['cell_line'] == cell_line]
        grp = sub.groupby('depth_um')['pearson_r']
        mean = grp.mean()
        sem  = grp.sem()
        ax.plot(mean.index, mean.values, ls=ls, label=cell_line, linewidth=2)
        ax.fill_between(mean.index, mean - sem, mean + sem, alpha=0.2)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(compartment)
    ax.set_xlabel('Depth (µm)')
    ax.set_ylabel('Pearson r' if ax == axes2[0] else '')
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial', 'Liberation Sans', 'DejaVu Sans']

compartments = ['cells', 'nuclei', 'cytoplasm']
treat_order  = ['DMSO', 'binimetinib', 'abemaciclib', 'SN-38']

depths     = sorted(df_corr_plane['depth_um'].unique())
cmap       = cm.get_cmap('gist_gray_r', len(depths))
depth_colors = {d: cmap(i) for i, d in enumerate(depths)}

fig, axes = plt.subplots(1, 3, figsize=(5, 3), sharey=True)

for ax, compartment in zip(axes, compartments):
    sub = df_corr_plane[df_corr_plane['compartment'] == compartment]

    for depth in depths:
        d_sub = sub[sub['depth_um'] == depth]
        means = d_sub.groupby('treatment')['pearson_r'].mean().reindex(treat_order)
        sems  = d_sub.groupby('treatment')['pearson_r'].sem().reindex(treat_order)

        ax.plot(treat_order, means.values, color=depth_colors[depth],
                linewidth=0.5, marker='o', markersize=3)
        ax.errorbar(treat_order, means.values, yerr=sems.values,
                    fmt='none', color=depth_colors[depth], capsize=2, linewidth=1)

    ax.set_title(compartment, fontsize=11, fontweight='bold')
    ax.set_xlabel('Treatment', fontsize=10)
    ax.set_ylabel('Pearson r' if ax == axes[0] else '', fontsize=10)
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='x', rotation=25, labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

# Colorbar for depth
norm = mcolors.Normalize(vmin=min(depths), vmax=max(depths))
sm   = cm.ScalarMappable(cmap='gist_gray_r', norm=norm)
sm.set_array([])
# cbar = fig.colorbar(sm, ax=axes[-1], shrink=0.8, pad=0.02)
# cbar.set_label('Depth (µm)', fontsize=9)

fig.suptitle('Pearson r (expert vs. cellpose) by treatment',
             fontsize=10, y=1.02)
plt.tight_layout()

save_panel(fig, 'SupplFig2e', data=df_corr_plane,
           caption='Feature correlation vs imaging depth, by compartment',
           notebook='analysis/3_SupplFigure2/error_propegation.ipynb')
plt.show()
